# What LLMs Do — Tokens, tiktoken and Context Windows

Your entire chunk in one sentence:

> **An LLM predicts the next piece of text, one piece at a time; those pieces are TOKENS, and the model's short-term memory for them is the CONTEXT WINDOW.**

Triage version (our mini-project):

> **Every email we classify and every reply we draft is bought and paid for in tokens. Count them, budget them, or the window fills up and the bill surprises you.**

After your talk (Monday round), the class must be able to:

1. Say what an LLM actually does (predicts next tokens).
2. **Count tokens** with tiktoken.
3. Explain how **context limits affect answers and cost**.
4. Sketch a **context budget** for one triage call.

Friday build: count tokens for 20 sample emails and estimate the context budget.

That is the test. Everything in this notebook serves those 4 goals.

## 1. Start from absolute zero: what is an LLM?

**LLM = Large Language Model.** Forget the hype. Mechanically it is one trick:

> **Read some text, guess the next piece. Append it. Guess again. Repeat until done.**

Your phone's keyboard does this (*Good... morning!*). An LLM is that same autocomplete idea with a gigantic memory of patterns learned from enormous piles of text — books, websites, code, manuals.

```text
  You send:   'Dear library team, I returned the book but got a fine...'
  Model thinks: probable next pieces... ' I' ... ' will' ... ' check' ...
  Model sends:  'I will check the loan record and update you shortly.'
```

Three words in the name, decoded:

| Word | Meaning |
|---|---|
| Large | Billions of adjustable knobs (parameters). More knobs ≈ subtler patterns. |
| Language | Input and output are text. It does not see your database; it sees words. |
| Model | A math machine: text in, probabilities out. |

> **Child version: an LLM is a parrot that read a million libraries and became scarily good at finishing your sentences. It does not KNOW things — it predicts word-shaped patterns.**

## 2. The prediction game, step by step

Inside one reply, this loop runs per piece:

```text
  1. LOOK at all text so far (your prompt + everything generated yet).
  2. SCORE every possible next piece (tens of thousands of candidates).
  3. PICK one (usually a likely one, with some randomness).
  4. APPEND it and go back to step 1.
```

Example. Prompt: *'The fine for a late book is'*. The model might score:

```text
  ' 500' .......... 34%
  ' calculated' .... 22%
  ' waived' ........ 8%
  ... thousands more tiny slices ...
```

It picks (say) ' 500', appends it, and now predicts what follows *'The fine for a late book is 500'* — maybe ' naira'. One piece at a time, until it emits a STOP piece.

Two consequences you must internalise:

1. **It cannot plan ahead.** It never sketches the full reply first — it walks forward greedily. That is why long precise outputs (JSON, reasoning) need care: Week 6 exists because of this.
2. **Randomness is a dial (temperature).** Low temperature = always pick the top piece (boring, reliable — what triage wants). High temperature = roll the dice (creative, wild). Member 8 will show the setting in the first real call; Member 6 compares its effects. For now: *triage = low temperature, always.*

> **One-liner for class: the model does not write answers — it gambles its way forward, one token at a time.**

## 3. What training actually does (60 seconds, no mystique)

Training = showing the machine trillions of text pieces and, each time it guesses badly, twisting its knobs slightly toward better guesses. Repeat for months on thousands of chips.

What comes OUT of training:

- **Patterns, not a database.** It never stores *'Ada owes 1,500 naira'* as a fact. It stores knob-positions that make fine-related sentences flow plausibly. Ask about YOUR library's data and it will happily invent an answer — that invention has a name: **hallucination**.
- **A frozen snapshot.** Training ends on some date (the **knowledge cutoff**). Events after that? Unknown. (Member 6 will note each model's cutoff when comparing.)
- **No memory of YOU.** Each API call starts blank. If you want it to know something, you must PUT it in the prompt — which spends tokens (your whole chunk!).

| Beginners believe | Reality |
|---|---|
| It looked it up | It predicted likely text |
| It remembers our last chat | Each API call is a stranger (unless YOU resend history = more tokens) |
| It knows today's news | It knows nothing past its cutoff |
| Confident = correct | Confidence is just fluency; verification is YOUR job (Weeks 6–7) |

> **The sentence that prevents 90% of LLM disasters: prediction is not knowledge. Everything the model says about YOUR world must be supplied by you or checked by you.**

## 4. What LLMs DO vs DON'T (triage edition)

Memorise this table. It draws the boundary of Mini-project 2.

**They DO (our triage relies on these):**

| Skill | Triage use |
|---|---|
| **Classify text** | *This email is a complaint / invoice / spam...* |
| **Extract fields** | Sender type, urgency, dates, amounts |
| **Summarise** | Long complaint → 2-line brief for the librarian |
| **Draft replies** | Polite 60-word draft a HUMAN reviews before sending |
| **Follow format instructions** | *Reply with exactly one category word* (mostly!) |
| **Translate / rephrase** | Rewrite a stiff reply warmly |

**They DON'T (design around these):**

| Limit | Triage consequence |
|---|---|
| Know YOUR data | We must paste the email into the prompt (tokens!) |
| Remember between calls | Every call resends instructions (tokens again!) |
| Guarantee format | Week 6 validates JSON; never trust raw text blindly |
| Do exact maths | Fine = 14 days × rate? Compute in PYTHON, never ask the model |
| Keep secrets | Never paste passwords/keys; Member 7 owns secret hygiene |
| Say 'I don't know' reliably | Member 4 writes explicit uncertainty rules |

> **Exam answer: LLMs predict fluent text from patterns. Triage uses them for reading and drafting, never for facts, maths, or final decisions — a human approves every reply.**

## 5. Meet the mini-project: Inbox Triage Assistant

Month 2 (Weeks 5–7) builds ONE thing:

> **An assistant that reads incoming emails, classifies each one, and drafts a reply for a HUMAN to review. The human approves, edits, or rejects. Nothing sends itself.**

One email's journey (Weeks 5→7):

```text
  Email arrives
      │
      ▼
  WEEK 5 (this week): send email + instructions to the model ──► raw text back
      │
      ▼
  WEEK 6: force/validate structured JSON (category, urgency, draft)
      │
      ▼
  WEEK 7: measure accuracy on a test set, write the report
      │
      ▼
  Human reviews the draft → approves or fixes → send
```

Categories (Week 6 formalises these as enums — peek, don't deep-dive): meeting request, invoice, complaint, newsletter, support request, spam. Plus urgency: low / normal / high.

Why the human gate matters: the model can misread tone (*'Your fine policy is a joke'* — complaint or humour?), invent policy, or confidently file an invoice as a newsletter. **The human is theeval gate of last resort** — a theme that returns in Weeks 12, 18 and 19. Say it early, say it often.

## 6. Why YOUR chunk is the foundation of the whole month

Every other Week 5 chunk SPENDS tokens. You teach the currency.

| Chunk | How it spends tokens | Needs from you |
|---|---|---|
| Member 2: system instructions | Every rule added to the prompt | *How big may the prompt be?* |
| Member 3: few-shot examples | Each example × its tokens | *Do 3 examples still fit?* |
| Member 4: uncertainty wording | Refusal/escalation text in + out | *Cost of the safety net?* |
| Member 6: model choice | Window size + price per 1K differ per model | *Tokens per email, so prices compare fairly* |
| Member 7: secrets | Redaction changes text length | *Re-count after redacting* |
| Member 8: first call + logging | Logs must record token use | *What to log: in/out counts* |

Say this on Monday with pride:

> **No other chunk can do its job until mine is done. Prompts cannot be designed, models cannot be compared, and bills cannot be predicted until someone can COUNT. I am that someone.**

Friday proves it: your token counts for 20 emails become the team's shared budget table that Members 2, 3 and 6 build on.

## 7. What is a token? (the heart of your chunk)

A **token** is one piece of text from the model's fixed menu of ~100,000 pieces (its **vocabulary**). The model reads tokens, thinks in tokens, writes tokens. It never sees 'words' — only these pieces.

Real splits from `tiktoken` (encoding `cl100k_base`, measured for this lesson — run them yourself!):

| Text | Tokens | Split | Surprise |
|---|---|---|---|
| `Hello world` | 2 | `Hello` + ` world` | The SPACE glues to `world`! |
| `Hello world!` | 3 | `Hello` + ` world` + `!` | One `!` = a whole token |
| `Port Harcourt` | 3 | `Port` + ` Har` + `court` | Rare words shatter |
| `user@example.com` | 3 | `user` + `@example` + `.com` | Emails are cheap-ish |
| `https://example.com/loans/42` | 9 | `https` + `://` + ... | URLs are EXPENSIVE |
| `supercalifragilistic...` | 11 | `sup`+`erc`+`al`+... | Long rare word = 11 tokens |
| `DON'T SHOUT` | 4 | `DON`+`'T`+` SH`+`OUT` | CAPS cost more |
| `😊` | 2 | (two byte-pieces) | One emoji = 2 tokens |

And a realistic triage sentence:

```text
Dear Ada, your book is due tomorrow. Please return it to avoid a fine.
→ 17 tokens (68 characters, 14 words)
```

> **The three facts to chant: tokens are NOT words. Tokens are NOT characters. Common text is cheap, rare text shatters.**

## 8. Why tokens exist (the trade-off in one paragraph)

Why not just use words? Or characters? Both fail:

| Idea | Problem |
|---|---|
| One menu entry per WORD | Millions of words exist (names, typos, new slang) — the menu would be enormous and slow |
| One entry per CHARACTER | Tiny menu, but `Hello` = 5 steps to generate — replies take forever |
| **Tokens (the compromise)** | ~100k pieces: common words get ONE piece (` the`), rare words split into a few (` Har`+`court`) |

How the menu is built (BPE, byte-pair encoding — know the NAME, not the maths): start with single characters, repeatedly merge the most frequent pair (*t+h → th, th+e → the*). Frequent text compresses to few tokens; rare text falls back to small fragments.

Consequences for triage:

1. **English library emails are cheap** — common words, ~1 token each.
2. **Names, codes, URLs, barcodes shatter** — `COPY-0001`, `https://...` cost several tokens each. (Member 4: refusal templates with URLs cost more than you think.)
3. **Other languages / scripts cost more** — non-English text fragments harder. A Yoruba or Pidgin email costs more tokens than its English twin. Budget with YOUR real emails (Friday!), not guesses.

> **Trade-off to quote (rule 6): a bigger vocabulary means fewer tokens per text but a bigger, slower model. Tokenisation is compression with a price tag on weirdness.**

## 9. Weird-but-true token rules (quiz the class on these!)

All verified with tiktoken for this lesson:

1. **Spaces attach FORWARD.** ` world` is one token (space included), not `world` + space. So *leading* spaces are nearly free, but each new word's space rides along.
2. **Shattering is gradual.** `1984` → 2 tokens (`198`+`4`). `10,000` → 3 (`10`+`,`+`000`). Numbers are NOT one-token-per-digit, nor one token — they chunk oddly. Never hand-estimate numbers; count them.
3. **CAPS and rare casing cost.** `DON'T SHOUT` = 4 tokens for 2 words. Fixed templates in ALL CAPS quietly inflate every call.
4. **Punctuation usually rides free-ish.** ` Sundays.` — the period often joins, but `!` split off in `Hello world!`. Inconsistent-looking, fully deterministic: same text + same encoding = same split, always.
5. **Whitespace has shape.** Three leading spaces became their own token (`  ` + ...). Indentation, blank lines, markdown tables all cost. Pretty prompts are pricier prompts (Member 2, take note).
6. **Same STRING, different ENCODING → different count.** `cl100k_base` (GPT-4-class) vs another model's encoding split differently. Rule: **always count with YOUR model's encoding.** Counting with the wrong ruler is worse than not counting — it lies confidently.

> **Class demo: write 5 guesses on the board (email? URL? emoji? number? shouty caps?), take bets, then run tiktoken LIVE. Gasps guaranteed. Nobody intuits tokenisation — that is WHY we measure.**

## 10. tiktoken: the ruler (install + first count)

**tiktoken** = the official token-counter library (made by OpenAI, open source). It contains the exact split-menus (encodings) so your counts match the model's.

```python
# pip install tiktoken   (uv add tiktoken)
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")  # the menu GPT-4-class models use

text = "Hello world"
ids = enc.encode(text)        # [15339, 1917]  (the menu item numbers)
print(len(ids))               # 2  <-- THE count
print([enc.decode([i]) for i in ids])  # ['Hello', ' world']  <-- the split
```

Vocabulary:

- **Encoding** (`cl100k_base`) = WHICH menu. `cl100k` ≈ 100k pieces. Different model families → different encodings → different counts. Member 6 tells you which encoding each shortlisted model uses; until then, `cl100k_base` is the classroom standard.
- **Token IDs** = the numbers (`15339`). The model eats numbers, not letters. Text → IDs is `encode`; IDs → text is `decode`.
- **Decode check:** always decode back once per new pipeline — if the text round-trips, your counting path is sane.

First run downloads the encoding file (~2 MB, cached). Offline afterwards. If install day has no network, pre-download. (Harbour rule: note it in the README's run instructions.)

## 11. Counting code you must be able to write cold

Three snippets. Type them until boring — Friday and the question round both assume them.

**Snippet 1: count one text.**

```python
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

def count(text: str) -> int:
    return len(enc.encode(text))

print(count("Dear Ada, your book is due tomorrow."))  # 10
```

**Snippet 2: count a whole batch (Friday's core loop).**

```python
emails = [...]  # 20 sample emails, fictional data only
counts = [count(e) for e in emails]
print("min:", min(counts), "max:", max(counts))
print("avg:", sum(counts) / len(counts))
```

**Snippet 3: count a full model call (prompt + expected reply).**

```python
system = "You are the Inbox Triage Assistant..."  # Member 2's instructions
email = "Subject: Overdue book fine..."          # one incoming email
prompt_tokens = count(system) + count(email) + 10  # +10: per-message overhead
reply_budget = 150   # room we RESERVE for the answer (max_tokens)
call_total = prompt_tokens + reply_budget
print("this call needs about", call_total, "tokens of window")
```

Notes:

- The `+10` is **message overhead**: role labels (`system`, `user`) and separators cost a few tokens per message. Exact recipe varies by model — Member 8 logs the provider's *reported* usage and we compare. Until then: small headroom beats false precision.
- `reply_budget` is not a guess — it becomes the API's `max_tokens` setting: the model is CUT OFF there. Budget says what we can afford; `max_tokens` enforces it.
- Fictional emails only (team rule 5). Real inboxes stay out of the repo.

## 12. The context window: short-term memory with a hard edge

> **The context window = the maximum tokens the model can look at in ONE call: everything you send IN plus everything it writes OUT, sharing one fixed space.**

The backpack analogy (draw it):

```text
  THE WINDOW (say 8,000 tokens — Member 6 confirms ours)
  +----------------------------------------------------------------+
  | system rules (75) | email (70) | examples (300) | REPLY SPACE (150) | ... free ... |
  +----------------------------------------------------------------+
  ▲                                                  ▲
  everything IN                                      everything OUT comes from here too
```

Rules of the backpack:

1. **In + out share it.** A 4,000-token email inside an 8,000 window leaves ~4,000 for instructions + reply — minus everything else you packed.
2. **Every call packs a FRESH backpack.** The model remembers NOTHING between calls. Chat history? YOU resend it each time (that's why long chats get slower and pricier — the backpack refills every turn).
3. **Sizes differ wildly.** Small/local models ≈ 8K; big hosted ones ≈ 128K+. Bigger sounds strictly better — section 14 explains why it isn't.
4. **Measure in tokens, never words.** 'Our emails average 200 words' tells you nothing until tokenised. Words→tokens varies ~1.0–2.0× by content. COUNT.

> **Exam sentence: the context window caps input PLUS output tokens per call; exceeding it breaks the call, so every feature (rules, examples, history) must fit inside a shared budget.**

## 13. What happens at the limit (three fates)

Fate 1: **The API refuses.** Prompt alone exceeds the window → error (400 / context-length error), no answer, but you still paid nothing for output. Your code MUST catch this and shrink-and-retry or escalate (Member 4's escalation path!).

Fate 2: **The reply is cut mid-sentence.** Prompt fits, but prompt + full reply would overflow → the model stops at the edge. JSON breaks in half (`{"category": "comp...`). Week 6 exists partly because of this: validate, detect truncation, retry with a bigger reply budget or smaller prompt.

Fate 3 (sneakiest): **It fits but the model forgets the middle.** With very long inputs, models attend sharply to the START and END and blur the MIDDLE ('lost in the middle'). A 100-email paste where the key complaint sits at email 50? Likely missed. Fixes: put key instructions LAST as well as first, retrieve only relevant text (that's all of Month 3: RAG!), keep prompts lean.

| Strategy when too big | How | Trade-off |
|---|---|---|
| Truncate | Keep first/last N tokens | May cut the crucial part |
| Summarise first | Compress email, then triage | Extra call = extra cost + possible lost detail |
| Retrieve (Month 3) | Send only relevant chunks | Needs retrieval machinery |
| Bigger-window model | Member 6's comparison | Pricier per token, slower, still forgets middles |

> **Triage policy to propose Friday: emails over our budget get truncated with a marker (`[TRUNCATED]`) + auto-escalated to human review. Never silently classify half an email.**

## 14. Tokens = MONEY (the bill, worked live)

Providers charge **per 1,000 (or 1M) tokens**, with INPUT and OUTPUT priced separately — output usually costs MORE (generation is the expensive part).

The formula (memorise; plug in YOUR model's prices from Member 6):

```text
  cost(call) = in_tokens × price_in + out_tokens × price_out
```

Worked example with CLEARLY-LABELED example prices (suppose $1 per 1M input, $3 per 1M output — NOT a quote!):

Real counts from section 8's method (measured!):

| Piece | Tokens |
|---|---|
| System rules | 75 |
| One email | 70 |
| Message overhead | ~10 |
| Reply (JSON + draft) | 39 |

```text
  in  = 75 + 70 + 10 = 155 tokens → 155 × $1/1M      = $0.000155
  out = 39 tokens                → 39 × $3/1M       = $0.000117
  one email ≈ $0.00027  (about three-hundredths of a cent)
```

Scale it (the question every manager asks):

```text
  20 emails/day × 30 days = 600 calls/month
  600 × $0.00027 ≈ $0.16/month for the pilot. Cheap!
```

NOW change one assumption and watch the bill move (this is the teaching moment):

- Add three 300-token examples (Member 3): in = 1,055 → **7× the input bill**.
- Allow 1,000-token replies instead of 150: output dominates.
- 10,000 emails/day production: $0.00027 × 300,000/month ≈ **$81/month** — suddenly worth optimising.

> **Mantras: output tokens cost more than input tokens. Few-shot examples are a subscription you pay on EVERY call. Small prompts are a money decision, not a style decision.**

## 15. Tokens = SPEED (why long prompts feel slow)

Two phases per call (one line each — Member 8 measures them Friday):

1. **Reading (prefill):** model digests your whole prompt before writing word one. Bigger prompt → longer silence before the first word. Grows steeply with input length.
2. **Writing (decode):** one token at a time, each waiting on the last. Longer reply → longer total wait, roughly linear. This is also why replies STREAM (you see words appear): the waiter brings dishes as cooked.

| Symptom | Cause | Fix |
|---|---|---|
| Slow FIRST word | Huge prompt / big model | Trim prompt; smaller model (Member 6) |
| Slow TOTAL reply | Long reply | Tight `max_tokens`; ask for brief JSON |
| Fine alone, slow at scale | Many calls × tokens | Batch off-peak; cache repeated prompts |

Triage targets to propose: p95 (95% of calls) under ~5s per email; log input/output tokens + latency on EVERY call (Member 8's JSON log) so Week 6 can draw the cost/latency curve.

> **Trade-off (rule 6): bigger model = smarter answers but slower + pricier per token. The whole game of Week 5 is finding the CHEAPEST model+prompt that still triages correctly — and everything starts from YOUR counts.**

## 16. Context budgeting: the envelope (put it all together)

A **context budget** = one table proving a call fits the window AND the wallet. Write one per prompt design. Here is triage's, with our REAL counts:

| Slot | Tokens | Notes |
|---|---:|---|
| System rules (Member 2) | 75 | Measured; grows with every rule added |
| Few-shot examples (Member 3) | 0 today / ~900 if 3-shot | ZERO-shot to start — examples are optional rent |
| Incoming email (your Friday max) | ≤ 500 | Cap: bigger → truncate + escalate |
| Message overhead | ~10 | Role labels + separators |
| **Input subtotal** | **≤ 585** | |
| Reply reservation (`max_tokens`) | 150 | JSON + 60-word draft; enforced cutoff |
| **Call total** | **≤ 735** | |
| Window (say 8K) | 8,000 | Headroom: 91% free → safe |
| Cost (example prices) | ≈ $0.001/call | 600 calls/mo ≈ $0.60 |

Budget RULES (chant these):

1. **Every slot has an owner.** System→M2, examples→M3, email cap→you, reply cap→you. Nobody spends without updating the table.
2. **Reserve output FIRST.** Set `max_tokens` from the budget, not from hope. Unreserved replies get truncated (Fate 2).
3. **Design for the MAX email, not the average.** Averages lie; one 3,000-token angry complaint breaks an average-sized window. Friday reports min/max/avg — the MAX sets the cap.
4. **Re-budget on every prompt change.** New rule? New example? Re-count, re-sum, re-check. Two minutes, saves production incidents.

> **Deliverable language for Friday: 'Budget v1: 735 tokens/call against an 8K window (9% used). Bottleneck: email cap 500. Next experiment: 3-shot adds ~900 in-tokens; approved only if accuracy (Week 7) justifies 7× input cost.' THAT is professional.**

## 17. Friday build walkthrough: count 20 emails, estimate the budget

Your exact Friday task. Rehearse until you can do it live and explain it before merge (team rule 1).

**Step 0 — fixtures:** 20 fictional emails covering all 6 categories + edge cases (very short, very long, angry + URL, non-English greeting, ALL CAPS subject). Fictional data only (rule 5). Save as `tests/fixtures/emails.json`.

**Step 1 — the counter (`scripts/count_tokens.py`):**

```python
import json
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

emails = json.load(open("tests/fixtures/emails.json"))
counts = sorted(len(enc.encode(e["body"])) for e in emails)
avg = sum(counts) / len(counts)
p95 = counts[int(len(counts) * 0.95) - 1]
print(f"n={len(counts)} min={counts[0]} max={counts[-1]} avg={avg:.0f} p95={p95}")
```

**Step 2 — read the numbers like a professional:**

- `max` sets the email cap (round UP with margin: max 412 → cap 500).
- `p95` tells you what 'normally big' looks like; `avg` is for COST maths only, never for caps.
- Flag outliers: the 2,000-token complaint gets the truncate+escalate policy (section 13), not a bigger budget.

**Step 3 — publish Budget v1** (section 16's table) in the mini-project README + PR: window assumed, per-slot tokens, $/call, $/month at pilot scale AND at 10k/day. Member 6 plugs in real prices; Member 8's logger will verify your estimates against reported usage.

**Step 4 — failure experiment (rule 4, pick this one):** craft one 10,000-token email, run it against the real call path, and SHOW the class what happens (error or truncation + your handler catching it). Screenshot the error AND the graceful escalation. That contrast is the lesson.

**What to say while demoing:** *Twenty fictional emails, counted with cl100k_base: min X, max Y, p95 Z. Cap set from max. Budget v1 fits 9% of an 8K window at ~$0.001/call. The monster email proves our guardrail works. AI-use log in the PR.*

## 18. Rules of thumb (tattoo these on your brain)

1. **1 token ≈ 4 characters ≈ 3/4 of an English word.** Estimation ONLY — always verify with tiktoken before promising.
2. **Count with YOUR model's encoding.** Wrong ruler, confident lies.
3. **In + out share ONE window.** Budget both; reserve output with `max_tokens`.
4. **Output costs more than input.** Brevity instructions (*'under 60 words'*) are money.
5. **Design caps from MAX, costs from AVG.** Averages never break production; maximums do.
6. **Re-count every prompt change.** Rules and examples are rented tokens, paid per call.
7. **Log tokens on every call.** Estimates are claims; Member 8's log is evidence.
8. **Weird text costs more.** URLs, codes, caps, emoji, non-English — measure YOUR real data.

> **The 10-second version for anyone who corners you: models read tokens, windows cap tokens, tokens cost money and time — so count everything, budget everything.**

## 19. Beginner traps + honest trade-offs

Team rules REQUIRE trade-offs in every answer (rule 6) and honest limits (rule 7). Memorise these.

### 6 traps beginners fall into

1. **Counting words, not tokens.** '200 words, plenty of room!' — then URLs and codes blow the budget. Fix: tiktoken, always.
2. **Forgetting the reply side.** Prompt fits beautifully; the 1,000-token answer overflows. Fix: reserve `max_tokens` in the budget FIRST.
3. **Ignoring per-message overhead.** 20 chat-history messages × a few tokens each = hundreds of invisible tokens. Fix: headroom per message; better, summarise history.
4. **Prompt bloat.** 40 rules 'just in case', 10 examples 'for safety'. Each is a per-call tax. Fix: budget table with owners; cut what Week 7 can't justify.
5. **'Bigger window solves everything'.** Bigger windows cost more, run slower, and still blur middles. Fix: lean prompts + (Month 3) retrieval.
6. **Trusting estimates over logs.** Estimates assume; providers report. Fix: compare YOUR counts vs the API's `usage` field (Member 8) and reconcile gaps publicly.

### 5 trade-offs to sound professional

| Choice | Gain | Cost |
|---|---|---|
| Zero-shot (no examples) | Cheapest, smallest prompt | Weaker format-following; Week 7 measures how much |
| Few-shot (3 examples) | Better accuracy + format | ~900 extra input tokens on EVERY call |
| Big window model | Swallows long emails whole | Pricier/slower per token; middles still blur |
| Tight max_tokens | Cheap, fast, forces brevity | Long legit replies truncate → must detect + retry |
| Verbose system rules | Safer behaviour | Permanent per-call token tax |

> **One-line honesty for your PR: counts use cl100k_base; verified against provider usage within ±X%; image/file inputs NOT counted by tiktoken — listed as a known limit.**

## 20. Your 20 question-round questions (with answers + trade-offs)

You run these on Monday. Each maps to your *after the talk* note; each carries a trade-off where one applies (rule 6).

1. **What does an LLM do, mechanically?** Predicts the next token from all text so far, appends it, repeats until STOP.
2. **Why can't it 'look things up'?** Training stores patterns (knob positions), not a fact database; our data must be pasted into prompts.
3. **Prediction vs knowledge — why does it matter?** Fluent ≠ true; triage verifies with JSON validation (W6), tests (W7), human approval. Trade-off: verification costs calls.
4. **Low vs high temperature?** Low = top pick, reliable (triage); high = creative dice. Members 6/8 demo it.
5. **What is a token?** One piece from the model's ~100k-piece menu. Not a word, not a character.
6. **Split `Hello world` and explain the space.** 2 tokens: `Hello` + ` world` — spaces attach FORWARD.
7. **Why does `Port Harcourt` cost 3 but `the` costs 1?** BPE gives frequent text single pieces; rare text shatters into fragments.
8. **Rank by tokens: email address, URL, emoji, shouty caps — why?** URL (9) > caps (4) > email (3) > emoji (2): rarity + fragmentation rule. Verify live!
9. **What is tiktoken?** The official counter library holding exact model menus (encodings) so counts match the model.
10. **Write the one-line counter.** `len(enc.encode(text))` with `enc = get_encoding("cl100k_base")`. Trade-off of guessing instead: confident lies.
11. **Why must the encoding match YOUR model?** Same string splits differently per menu; wrong ruler, wrong budget, burst window.
12. **What shares the context window?** Input AND output tokens in one call, plus per-message overhead. Every call packs fresh.
13. **Three fates at the limit?** API refuses; reply truncates mid-JSON; model blurs the middle. Fix menu: truncate/summarise/retrieve/bigger window.
14. **State the cost formula.** `in×price_in + out×price_out`. Output usually pricier; prices from Member 6.
15. **One triage call ≈ how much (our numbers)?** ~155 in + 39 out ≈ $0.00027 at example prices; 600/mo ≈ $0.16 pilot.
16. **What does 3-shot do to that bill?** ~900 more input tokens ≈ 7× input cost, every call. Worth it only if Week 7 accuracy says so.
17. **Why do long prompts feel slow?** Reading phase delays the first word; writing is one-token-at-a-time. Trim prompt / cap reply.
18. **Read the budget table: who owns each slot?** System→M2, examples→M3, email cap→you, reply cap→you. Re-budget every prompt change.
19. **MAX or AVG for caps — and why?** MAX sets caps (maximums break production); AVG only for cost maths.
20. **Your Friday deliverable in one breath?** 20 fictional emails counted (min/max/avg/p95), cap from max, Budget v1 table, monster-email failure demo, AI-use log.

Run them in order 1-20: mechanism → limits → triage → tokens → tiktoken → window → money → speed → budget → Friday. That order IS your lesson arc.

## 22. Cheat sheet

```text
LLM: predicts next token, appends, repeats. Patterns, not knowledge. Blank every call.
TRIAGE: model classifies + drafts; human approves. Never facts/maths/final calls.
TOKEN: one menu piece (~100k menu). NOT words/chars. Common=1, rare=shatters.
  spaces attach forward (' world'). caps/URLs/codes/emoji/non-English cost extra.
TIKTOKEN: len(enc.encode(text)), encoding MUST match your model (cl100k_base here).
WINDOW: in + out + overhead share ONE cap per call. Overflow = error/truncate/blur-middle.
MONEY: in x price_in + out x price_out. Output pricier. Examples = per-call rent.
SPEED: big prompt delays first word; long reply delays finish. Trim + cap.
BUDGET: every slot owned; reserve reply FIRST (max_tokens); caps from MAX, costs from AVG.
OATH: count everything, budget everything, reserve the reply!
```